<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c38-header.png" alt="Nextia Learning · Search and Ranking: Find the Best Match" width="100%">

# Learn from clicks

**[Learn from clicks](https://learning.nextia-ai.com/courses/search/m06/learn-from-clicks/)** · Search and Ranking: Find the Best Match · Module 6, lesson 4 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** measure position bias in a query log, estimate how often people look at each position, correct clicks with inverse propensity weighting, and see what no correction can fix.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells (about 2 MB). No account, no key and no model download. |
| **You should know** | Learning to rank, from [Learning to rank](https://learning.nextia-ai.com/courses/search/m06/learning-to-rank/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1 and matplotlib 3.11.2 (bm25s 0.3.16, PyStemmer 3.1.0, sentence-transformers 6.2.0, PyTorch 2.15.0, scikit-learn 1.9.1 and LightGBM 4.7.0 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/search/m06/learn-from-clicks/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C38/M06-L04-learn-from-clicks/learn-from-clicks-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'pandas']:
    print(name, version(name))

## Setup: get the Larkfield data

Run the next cell. It downloads the files that this lesson needs (about 2 MB) into a folder `larkfield-search`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-search` (in Colab: the **Files** panel on the left), then run the cell again.

Where the data comes from: Larkfield is an invented shop. Its products, queries and relevance judgments are derived from WANDS, the Wayfair ANnotation DataSet (MIT licence). The query log with its clicks, and the prices, stock and sale flags, are **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C38/data/dataset-card.md) says what is real and what is made.

In [ ]:
# Setup: download the files into larkfield-search/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C38/"
FILES = {
    "data/queries.csv": "51e0286150a6162502faf6502e90b89e4984949ddcb64f1e83f7e1d3251d8634",
    "data/judgments.csv.gz": "6a0b896c64a0e1000fe9401bfc359ae7995732af458120963408d71e50333ec6",
    "data/log/searches.csv.gz": "73da1fb11cebb3891cabca5ab320be2147d1855b9f8e53890007b19fb78862ea",
    "data/log/impressions.csv.gz": "588cacc6c4ab6ac226478013644bf64871c69c32650fefd0219df431a38d80d0",
    "compare.py": "f09c610e8e9026e0c355f693cb56cd3e178e7217c0e001a744280fb82ff867d2",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-search").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-search and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-search.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, matplotlib, and the course helper as `cp` (NDCG, recall, MRR, intervals and the comparison table). It defines a few small helpers:

- `catalogue()`, `queries()` and `judgments()` read the data: one row per product, one row per query, and `{query_id: {product_id: grade}}` with grade 2 exact, 1 partial, 0 irrelevant.
- `show(ids)` prints products with their title and class.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import compare as cp
warnings.filterwarnings("ignore")

def catalogue():
    """One row per product."""
    df = pd.read_csv("data/catalogue.csv.gz", keep_default_na=False, dtype={"title_fr": str})
    for col in ("on_sale", "in_stock"):
        df[col] = df[col].astype(str).eq("True")
    return df

def queries():
    """One row per query with judgments (train, validation and test)."""
    return pd.read_csv("data/queries.csv", keep_default_na=False)

def judgments():
    """{query_id: {product_id: grade}}: 2 exact, 1 partial, 0 irrelevant."""
    j = pd.read_csv("data/judgments.csv.gz")
    out = {}
    for q, p, g in zip(j["query_id"], j["product_id"], j["grade"]):
        out.setdefault(q, {})[p] = int(g)
    return out

def show(ids, judged=None, n=10):
    """Print products: rank, ID, title, class (and the grade when judged is given)."""
    global _CAT
    if "_CAT" not in globals():
        _CAT = catalogue().set_index("product_id")
    cat = _CAT
    names = {2: "exact", 1: "partial", 0: "irrelevant"}
    for i, p in enumerate(list(ids)[:n], 1):
        g = "" if judged is None else f"  [{names.get(judged.get(p), 'not judged')}]"
        print(f"{i:2}. {p}  {cat.loc[p, 'title'][:55]:<55}  {cat.loc[p, 'product_class'][:22]}{g}")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Clicks fall with position

Every search in Larkfield's query log records the ten results it showed and what people clicked. The log is **synthetic**: our code made the clicks with a **click model**, a small set of rules for how people look and click (see the lesson). Run the next cell.

In [ ]:
q = queries()
j = judgments()
s = pd.read_csv("data/log/searches.csv.gz", keep_default_na=False)   # one row per search (synthetic)
imp = pd.read_csv("data/log/impressions.csv.gz")                     # one row per result shown
imp = imp.merge(s[["search_id", "query_id"]], on="search_id")
imp["grade"] = [j.get(a, {}).get(b, -1) if a else -1 for a, b in zip(imp["query_id"], imp["product_id"])]
print(len(s), "searches,", len(imp), "results shown")
ctr = imp.groupby("position")["clicked"].mean()
ctr.plot(marker="o", figsize=(6, 3), title="Click rate by position (synthetic log)")
plt.xlabel("position on the page"); plt.ylabel("click rate"); plt.grid(alpha=0.3); plt.show()
print(ctr.round(3).to_string())

> **Read the chart.** The first result gets a click 0.446 of the time, the tenth 0.041. Is the first result ten times more relevant? Or do people simply look at it more?

## 2. Separate position from relevance

To see position alone, keep relevance fixed: look only at exact products. If an exact product at position 4 gets fewer clicks than an exact product at position 1, the difference comes from position. This is **position bias**.

In [ ]:
# Only the exact products: the same relevance at every position, so what is left is position.
exact = imp[imp["grade"] == 2].groupby("position")["clicked"].agg(["size", "mean"])
exact.columns = ["exact shown", "click rate of exact"]
exact

## 3. Your turn: estimate examine(k)

The chance that a person looks at position k is called **examine(k)**. Estimate it: divide the click rate of exact products at each position by the rate at position 1.

In [ ]:
examine = ...   # the click rate of exact products at each position divided by the rate at position 1
print(examine)

In [ ]:
expect_hash('examine', examine, '970379176e1fcf0b')

> **Check.** examine(2) is about 0.52, examine(3) about 0.335 and examine(10) about 0.098. Our code made the log with examine(k) = 1/k (0.5, 0.333 … 0.1), and the estimate recovers it from the clicks alone. With real people you do not know the rule; this method estimates it from your own log.

## 4. Correct the clicks: inverse propensity weighting

A click at position 4 is stronger evidence than a click at position 1, because few people look at position 4. **Inverse propensity weighting** (IPW) divides each click by examine(k) of its position.

In [ ]:
# A tiny example: two products, each shown 4 times and clicked once.
toy = pd.DataFrame({"product": ["X", "Y"], "position": [1, 4], "shown": [4, 4], "clicks": [1, 1]})
toy["naive label"] = toy["clicks"] / toy["shown"]
toy["examine(k) = 1/k"] = 1 / toy["position"]
toy["IPW label"] = toy["naive label"] / toy["examine(k) = 1/k"]
toy

Both products got 1 click in 4. The naive label says they are equal. IPW says Y is four times stronger: it got the same clicks with a quarter of the attention.

## 5. Your turn: what the log never shows

IPW corrects for products that were shown low. It cannot correct for products that were **never shown**. How many exact products of the training queries did Larkfield's search ever show?

In [ ]:
train = q[q["split"] == "train"]
exact_pairs = {(a, b) for a in train["query_id"] for b, g in j.get(a, {}).items() if g == 2}
shown_pairs = set(zip(imp["query_id"], imp["product_id"]))
shown_share = ...   # the share of exact_pairs that are also in shown_pairs
print(len(exact_pairs), shown_share)

In [ ]:
expect('shown_share', shown_share, 0.04027056787792984, tolerance=0.0005)

In [ ]:
rows = []
for src, part in train.groupby("source"):
    pairs = {(a, b) for a in part["query_id"] for b, g in j.get(a, {}).items() if g == 2}
    rows.append({"source": src, "exact pairs": len(pairs), "ever shown": len(pairs & shown_pairs)})
pd.DataFrame(rows)

> **Check.** Only about 4% of the 12714 exact pairs were ever shown. Today's search needs every word to match, so French queries, misspellings and codes show nothing at all. A product that is never shown gets no click, whatever its quality. This is **selection bias**, and no click model can undo it. In the project: `search clicks bias`, then `search ltr train --labels clicks-ipw`.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Autocomplete](https://learning.nextia-ai.com/courses/search/m07/autocomplete/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/search/m06/learn-from-clicks/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The products, queries and judgments come from WANDS (Wayfair, MIT licence). The query log, prices, stock and sale flags are synthetic: our code made them. No real customer is in this data.